# Speech-to-Retrieval (S2R) — Entrainement sur Colab / Kaggle

**Pipeline complet : audio brut → embeddings → dual encoder → recherche documentaire**

Ce notebook execute les etapes suivantes :
1. Installation des dependances
2. Montage des donnees (Google Drive / upload / Kaggle)
3. Validation des donnees P1/P2
4. Pre-calcul des embeddings texte (MiniLM)
5. Entrainement du Dual Encoder (GPU)
6. Export des embeddings texte projetes
7. Test d'inference
8. Telechargement du checkpoint

---
**Temps estime sur GPU T4 (Colab gratuit) :** ~15-25 min pour 20 epoques  
**Donnees requises (~70 MB) :**
- `embeddings/audio_embeddings.npy`
- `embeddings/audio_embeddings_index.csv`
- `data/output/corpus_chunks.csv`
- `data/output/pairs_train.csv`
- `data/output/pairs_val.csv`
- Dossiers `src/`, `training/`

## Cellule 0 — Detection de la plateforme

In [ ]:
import os, sys

# Detection automatique de la plateforme
IS_COLAB  = 'google.colab' in sys.modules or os.path.exists('/content')
IS_KAGGLE = os.path.exists('/kaggle')

if IS_COLAB:
    print('Plateforme : Google Colab')
    ROOT = '/content/s2r'
elif IS_KAGGLE:
    print('Plateforme : Kaggle')
    ROOT = '/kaggle/working/s2r'
else:
    print('Plateforme : locale')
    ROOT = os.path.abspath('..')

print(f'Repertoire de travail : {ROOT}')

## Cellule 1 — Installation des dependances

In [ ]:
# Installation des packages requis
# Colab et Kaggle ont deja torch installe — on installe uniquement les packages manquants

!pip install -q \
    transformers==4.40.2 \
    sentence-transformers==3.0.0 \
    faiss-cpu \
    soundfile \
    torchaudio \
    tqdm \
    pandas \
    numpy

print('Installation terminee.')

In [ ]:
# Verification GPU
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA disponible : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device utilise : {DEVICE}')

## Cellule 2 — Chargement du code source

**Choisir UNE SEULE option :** A (GitHub), B (Google Drive), ou C (Upload direct)

In [ ]:
# ===========================================================================
# OPTION A : Cloner depuis GitHub (remplacer par votre URL)
# ===========================================================================
# Decommentez ce bloc si votre code est sur GitHub

# import os
# os.makedirs(ROOT, exist_ok=True)
# !git clone https://github.com/VOTRE_USERNAME/inpt_audioprocessing.git {ROOT}
# %cd {ROOT}

# ===========================================================================
# OPTION B : Monter Google Drive (Colab uniquement)
# ===========================================================================
# Le projet doit etre dans MonDrive/s2r/
# Structure attendue : MonDrive/s2r/src/, MonDrive/s2r/training/, etc.

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Chemin vers votre projet dans Drive — modifier si necessaire
    DRIVE_PROJECT = '/content/drive/MyDrive/s2r'
    
    import os
    if os.path.exists(DRIVE_PROJECT):
        ROOT = DRIVE_PROJECT
        print(f'Projet trouve dans Drive : {ROOT}')
    else:
        print(f'[WARN] Projet non trouve dans {DRIVE_PROJECT}')
        print('Creez le dossier s2r dans votre Drive et uploadez le projet')
        print('Ou utilisez OPTION C (upload direct ci-dessous)')

# ===========================================================================
# OPTION C : Upload direct (archive zip)
# ===========================================================================
# Decommentez ce bloc pour uploader une archive .zip

# from google.colab import files
# import zipfile, os
# print('Selectionnez le fichier s2r_upload.zip...')
# uploaded = files.upload()
# zip_name = list(uploaded.keys())[0]
# os.makedirs(ROOT, exist_ok=True)
# with zipfile.ZipFile(zip_name, 'r') as z:
#     z.extractall(ROOT)
# print(f'Archive extraite dans {ROOT}')

# ===========================================================================
# OPTION D : Kaggle Dataset
# ===========================================================================
# Si vous avez uploade le projet comme dataset Kaggle,
# le chemin d'entree est /kaggle/input/NOM_DU_DATASET/

# if IS_KAGGLE:
#     KAGGLE_INPUT = '/kaggle/input/s2r-project'  # Modifier avec votre nom de dataset
#     import shutil, os
#     os.makedirs(ROOT, exist_ok=True)
#     shutil.copytree(KAGGLE_INPUT, ROOT, dirs_exist_ok=True)
#     print(f'Dataset copie vers {ROOT}')

print(f'ROOT = {ROOT}')

In [ ]:
# Ajouter le projet au PYTHONPATH
import sys, os

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

os.chdir(ROOT)
print(f'Repertoire courant : {os.getcwd()}')

# Verifier la structure du projet
required = [
    'src/speech_encoder.py',
    'training/train_dual_encoder.py',
    'training/precompute_text.py',
    'training/export_text_embeddings.py',
    'training/validate_handoff.py',
]
missing = [f for f in required if not os.path.exists(f)]
if missing:
    print(f'[ERREUR] Fichiers manquants : {missing}')
    print('Verifiez que le projet est correctement charge (Cellule 2)')
else:
    print('[OK] Structure du projet correcte')

## Cellule 3 — Verification des donnees

Les fichiers de donnees doivent etre presents. Si vous utilisez Google Drive, ils sont deja la.
Sinon, uploadez-les ci-dessous.

In [ ]:
import os

# Fichiers de donnees requis
DATA_FILES = {
    'audio_embeddings'     : 'embeddings/audio_embeddings.npy',
    'audio_manifest'       : 'embeddings/audio_embeddings_index.csv',
    'corpus_chunks'        : 'data/output/corpus_chunks.csv',
    'pairs_train'          : 'data/output/pairs_train.csv',
    'pairs_val'            : 'data/output/pairs_val.csv',
}

all_ok = True
for name, path in DATA_FILES.items():
    full = os.path.join(ROOT, path)
    if os.path.exists(full):
        size = os.path.getsize(full) / 1e6
        print(f'[OK]  {path} ({size:.1f} MB)')
    else:
        print(f'[MANQUANT] {path}')
        all_ok = False

if not all_ok:
    print('\n[ACTION REQUISE] Uploadez les fichiers manquants (voir cellule suivante)')
else:
    print('\n[OK] Toutes les donnees sont presentes')

In [ ]:
# OPTIONNEL : Upload des fichiers de donnees manquants (Colab)
# Decommentez si des fichiers sont manquants et que vous n'utilisez pas Drive

# from google.colab import files
# import zipfile, os, shutil
#
# print('Uploadez le fichier data.zip contenant les fichiers manquants...')
# uploaded = files.upload()
# for fname, content in uploaded.items():
#     if fname.endswith('.zip'):
#         with zipfile.ZipFile(fname, 'r') as z:
#             z.extractall(ROOT)
#         print(f'Archive {fname} extraite')
#     else:
#         dest = os.path.join(ROOT, fname)
#         os.makedirs(os.path.dirname(dest), exist_ok=True)
#         with open(dest, 'wb') as f:
#             f.write(content)
#         print(f'Fichier {fname} sauvegarde')

## Etape 1 — Validation des donnees P1/P2

In [ ]:
# Validation des embeddings audio et des paires d'entrainement
# Attendu : manifest_rows=4985, embedding_shape=(4985, 768), pairs_rows=3988

!python -m training.validate_handoff \
    --audio-manifest-csv  embeddings/audio_embeddings_index.csv \
    --audio-embeddings-npy embeddings/audio_embeddings.npy \
    --pairs-csv           data/output/pairs_train.csv

## Etape 2 — Pre-calcul des embeddings texte (MiniLM)

Cette etape encode tous les chunks de texte avec MiniLM une seule fois.  
Elle n'est necessaire qu'une seule fois — les resultats sont sauvegardes dans `embeddings/`.

In [ ]:
import os

PRECOMPUTED_NPY      = 'embeddings/precomputed_text.npy'
PRECOMPUTED_MANIFEST = 'embeddings/precomputed_text_manifest.csv'

if os.path.exists(PRECOMPUTED_NPY):
    import numpy as np
    shape = np.load(PRECOMPUTED_NPY).shape
    print(f'[SKIP] Embeddings texte deja calcules : {PRECOMPUTED_NPY} {shape}')
else:
    print('Pre-calcul des embeddings texte (MiniLM)...')
    !python -m training.precompute_text \
        --text-chunks-csv data/output/corpus_chunks.csv \
        --model-name      sentence-transformers/all-MiniLM-L6-v2 \
        --output-npy      {PRECOMPUTED_NPY} \
        --output-manifest {PRECOMPUTED_MANIFEST} \
        --batch-size      128

## Etape 3 — Entrainement du Dual Encoder

**Mode rapide** : seules les couches de projection (2 FC layers) sont entrainées.  
Wav2Vec2 et MiniLM sont geles — pas de forward pass lourd pendant l'entrainement.

In [ ]:
# Hyperparametres d'entrainement
# Modifier selon vos besoins et les ressources disponibles

EPOCHS        = 20       # 20 epoques recommandees
BATCH_SIZE    = 64       # 64 sur GPU T4, reduire si OOM
PROJECTION_DIM = 256     # dimension de l'espace partage
LEARNING_RATE = '2e-4'   # plus eleve car projection uniquement
WARMUP_STEPS  = 100
LOSS          = 'contrastive'
TEMPERATURE   = 0.07
OUTPUT_DIR    = 'models/dual_encoder_mpnet'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output : {OUTPUT_DIR}')
print(f'Epochs : {EPOCHS} | Batch : {BATCH_SIZE} | Projection : {PROJECTION_DIM}D')

In [ ]:
# Lancement de l'entrainement
# Sur GPU T4 : ~1-2 min/epoque -> 20-40 min total

!python -m training.train_dual_encoder \
    --precomputed-text-npy      embeddings/precomputed_text.npy \
    --precomputed-text-manifest embeddings/precomputed_text_manifest.csv \
    --pairs-csv            data/output/pairs_train.csv \
    --val-pairs-csv        data/output/pairs_val.csv \
    --audio-manifest-csv   embeddings/audio_embeddings_index.csv \
    --audio-embeddings-npy embeddings/audio_embeddings.npy \
    --output-dir           {OUTPUT_DIR} \
    --projection-dim       {PROJECTION_DIM} \
    --epochs               {EPOCHS} \
    --batch-size           {BATCH_SIZE} \
    --learning-rate        {LEARNING_RATE} \
    --warmup-steps         {WARMUP_STEPS} \
    --loss                 {LOSS} \
    --temperature          {TEMPERATURE} \
    --num-workers          2

In [ ]:
# Verification du checkpoint sauvegarde
import os

CHECKPOINT = f'{OUTPUT_DIR}/best_model.pt'
if os.path.exists(CHECKPOINT):
    size = os.path.getsize(CHECKPOINT) / 1e6
    print(f'[OK] Checkpoint sauvegarde : {CHECKPOINT} ({size:.1f} MB)')
    
    import torch
    ckpt = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
    print(f'Config : {ckpt["config"]}')
    print(f'Metriques : {ckpt.get("best_metrics", {})}')
else:
    print(f'[ERREUR] Checkpoint introuvable : {CHECKPOINT}')

## Etape 4 — Export des embeddings texte projetes

Applique la couche `text_projection` entrainee sur tous les chunks de texte.  
Produit les embeddings 256D qui alimentent l'index FAISS.

In [ ]:
TEXT_CHUNK_EMBEDDINGS = 'embeddings/text_chunk_embeddings.npy'
TEXT_CHUNK_MANIFEST   = 'embeddings/text_chunk_manifest.csv'

!python -m training.export_text_embeddings \
    --precomputed-text-npy      embeddings/precomputed_text.npy \
    --precomputed-text-manifest embeddings/precomputed_text_manifest.csv \
    --text-chunks-csv           data/output/corpus_chunks.csv \
    --checkpoint                {CHECKPOINT} \
    --output-embeddings-npy     {TEXT_CHUNK_EMBEDDINGS} \
    --output-manifest-csv       {TEXT_CHUNK_MANIFEST} \
    --batch-size                128

In [ ]:
# Verification des embeddings exportes
import numpy as np, pandas as pd

embs = np.load(TEXT_CHUNK_EMBEDDINGS)
mfst = pd.read_csv(TEXT_CHUNK_MANIFEST)
print(f'Embeddings : {embs.shape}   (attendu : (N, {PROJECTION_DIM}))')
print(f'Manifest   : {len(mfst)} lignes')
print(mfst.head(3))

## Etape 5 — Test d'inference

Recherche documentaire sur un fichier audio exemple.

In [ ]:
# Lister les fichiers audio de requete disponibles
import glob, os

query_dir = 'data/audio_queries'
wav_files = sorted(glob.glob(f'{query_dir}/*.wav'))

if wav_files:
    print(f'{len(wav_files)} fichiers audio trouves :')
    for f in wav_files[:5]:
        print(f'  {f}')
    SAMPLE_WAV = wav_files[0]
    print(f'\nFichier de test : {SAMPLE_WAV}')
else:
    print('[WARN] Aucun fichier .wav dans data/audio_queries/')
    SAMPLE_WAV = None

In [ ]:
# Inference : audio -> top-5 documents
if SAMPLE_WAV:
    !python inference.py \
        --audio           "{SAMPLE_WAV}" \
        --checkpoint      {CHECKPOINT} \
        --text-embeddings {TEXT_CHUNK_EMBEDDINGS} \
        --manifest        {TEXT_CHUNK_MANIFEST} \
        --k               5
else:
    print('[SKIP] Aucun fichier audio disponible')

In [ ]:
# Inference par code Python (sans CLI)
import sys, os
sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import torch
import faiss

from src.speech_encoder import SpeechEncoder, _load_wav

def run_inference(audio_path: str, checkpoint: str, text_emb_path: str,
                  manifest_path: str, k: int = 5, device: str = DEVICE):
    """Inference complete : audio -> top-k chunks de texte."""
    from training.models import DualEncoderModel
    
    # Charger le dual encoder
    ckpt = torch.load(checkpoint, map_location='cpu', weights_only=False)
    model = DualEncoderModel(**ckpt['config'])
    model.load_state_dict(ckpt['state_dict'])
    model.eval().to(device)
    
    # Charger le speech encoder
    speech_enc = SpeechEncoder(frozen=True).eval().to(device)
    
    # Encoder l'audio
    waveform = _load_wav(audio_path).unsqueeze(0).to(device)
    with torch.no_grad():
        audio_emb = speech_enc(waveform)        # [1, 768]
        query_emb = model.encode_audio(audio_emb)  # [1, 256]
    query_vec = query_emb.squeeze(0).cpu().numpy().astype('float32')
    
    # Normaliser et rechercher
    embs = np.load(text_emb_path).astype('float32')
    norms = np.linalg.norm(embs, axis=1, keepdims=True)
    embs /= np.clip(norms, 1e-9, None)
    
    index = faiss.IndexFlatIP(embs.shape[1])
    index.add(embs)
    
    q = query_vec / max(np.linalg.norm(query_vec), 1e-9)
    scores, indices = index.search(q.reshape(1, -1), k)
    
    manifest = pd.read_csv(manifest_path)
    results = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), 1):
        row = manifest.iloc[int(idx)]
        results.append({
            'Rang': rank,
            'Score': round(float(score), 4),
            'chunk_id': str(row.get('chunk_id', '')),
            'Texte': str(row.get('text', row.get('prompt', '')))[:200],
        })
    return pd.DataFrame(results)


if SAMPLE_WAV and os.path.exists(CHECKPOINT):
    df = run_inference(
        audio_path    = SAMPLE_WAV,
        checkpoint    = CHECKPOINT,
        text_emb_path = TEXT_CHUNK_EMBEDDINGS,
        manifest_path = TEXT_CHUNK_MANIFEST,
        k             = 5,
    )
    print(df.to_string(index=False))
else:
    print('[SKIP] Donnees manquantes pour l\'inference')

## Etape 6 — Evaluation des metriques

In [ ]:
# Afficher les metriques enregistrees dans le checkpoint
import torch

if os.path.exists(CHECKPOINT):
    ckpt = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
    metrics = ckpt.get('best_metrics', {})
    
    print('=== Metriques du meilleur modele ===')
    print(f'Recall@5  : {metrics.get("Recall@5",  0):.4f}   (objectif : > 0.30)')
    print(f'Recall@10 : {metrics.get("Recall@10", 0):.4f}   (objectif : > 0.45)')
    print(f'MRR       : {metrics.get("MRR",       0):.4f}   (objectif : > 0.25)')
    
    if metrics.get('MRR', 0) < 0.10:
        print('\n[HINT] Metriques faibles. Essayez :')
        print('  - Augmenter EPOCHS a 30-50')
        print('  - Augmenter BATCH_SIZE (plus de negatifs in-batch)')
        print('  - Reduire TEMPERATURE (0.05 au lieu de 0.07)')
        print('  - Learning rate : 5e-4')
else:
    print('[SKIP] Checkpoint introuvable')

## Etape 7 — Telechargement du checkpoint

Recuperez les fichiers pour les utiliser en local.

In [ ]:
# ===========================================================================
# OPTION A : Sauvegarder vers Google Drive (Colab)
# ===========================================================================

if IS_COLAB:
    import shutil, os
    
    DRIVE_OUTPUT = '/content/drive/MyDrive/s2r_output'
    os.makedirs(DRIVE_OUTPUT, exist_ok=True)
    
    files_to_save = [
        (CHECKPOINT,               f'{DRIVE_OUTPUT}/best_model.pt'),
        (TEXT_CHUNK_EMBEDDINGS,    f'{DRIVE_OUTPUT}/text_chunk_embeddings.npy'),
        (TEXT_CHUNK_MANIFEST,      f'{DRIVE_OUTPUT}/text_chunk_manifest.csv'),
        (PRECOMPUTED_NPY,          f'{DRIVE_OUTPUT}/precomputed_text.npy'),
        (PRECOMPUTED_MANIFEST,     f'{DRIVE_OUTPUT}/precomputed_text_manifest.csv'),
    ]
    
    for src, dst in files_to_save:
        if os.path.exists(src):
            shutil.copy2(src, dst)
            size = os.path.getsize(dst) / 1e6
            print(f'[OK] Sauvegarde : {dst} ({size:.1f} MB)')
        else:
            print(f'[SKIP] Introuvable : {src}')

    print(f'\nFichiers sauvegardes dans : {DRIVE_OUTPUT}')

In [ ]:
# ===========================================================================
# OPTION B : Telechargement direct (Colab)
# ===========================================================================
# Decommentez pour telecharger les fichiers directement

# if IS_COLAB:
#     from google.colab import files
#     files.download(CHECKPOINT)
#     files.download(TEXT_CHUNK_EMBEDDINGS)
#     files.download(TEXT_CHUNK_MANIFEST)

In [ ]:
# ===========================================================================
# OPTION C : Creer une archive zip et sauvegarder / telecharger
# ===========================================================================

import zipfile, os

OUTPUT_ZIP = 's2r_trained_model.zip'

files_to_zip = [
    CHECKPOINT,
    TEXT_CHUNK_EMBEDDINGS,
    TEXT_CHUNK_MANIFEST,
    PRECOMPUTED_NPY,
    PRECOMPUTED_MANIFEST,
]

with zipfile.ZipFile(OUTPUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in files_to_zip:
        if os.path.exists(f):
            arcname = os.path.relpath(f, ROOT) if f.startswith(ROOT) else f
            zf.write(f, arcname)
            print(f'  + {arcname}')

total_size = os.path.getsize(OUTPUT_ZIP) / 1e6
print(f'\nArchive creee : {OUTPUT_ZIP} ({total_size:.1f} MB)')

# Decommentez pour telecharger (Colab)
# from google.colab import files
# files.download(OUTPUT_ZIP)

## Resume et prochaines etapes

| Etape | Statut |
|-------|--------|
| Validation P1/P2 | ✓ |
| Pre-calcul embeddings texte | ✓ |
| Entrainement Dual Encoder | ✓ |
| Export embeddings projetes | ✓ |
| Inference | ✓ |

**Pour utiliser le modele en local :**
1. Telecharger `best_model.pt`, `text_chunk_embeddings.npy`, `text_chunk_manifest.csv`
2. Placer dans les dossiers `models/dual_encoder_mpnet/` et `embeddings/`
3. Lancer : `python demo/app.py --checkpoint models/dual_encoder_mpnet/best_model.pt`

**Pour ameliorer les metriques :**
- Augmenter `EPOCHS` a 30-50 (GPU requis)
- Augmenter `BATCH_SIZE` a 128 (plus de negatifs in-batch ameliore la perte contrastive)
- Essayer `TEMPERATURE = 0.05`

**Objectifs apres 20 epoques :**
- Recall@5 > 0.30
- Recall@10 > 0.45  
- MRR > 0.25